# 🔬 Breast Cancer Classifier — Google Colab
> **Université de Thiès · UFR SET · MaRT2 · 2025-2026**  
> Pr. Cheikh SARR — Groupe A : Cheikh Awa Balla GUEYE & Nafissata THIAM

Ce notebook exécute la **chaîne complète** du projet :
1. Montage Google Drive
2. Installation des dépendances
3. Vérification GPU
4. Exploration du dataset
5. Prétraitement & DataLoaders
6. Entraînement EfficientNet-B3
7. Évaluation complète
8. Explicabilité Grad-CAM

---
⚠️ **Avant de commencer** : active le GPU  
`Exécution → Modifier le type d'exécution → GPU (T4) → Enregistrer`


## ⚙️ Étape 0 — Vérification du GPU
> Active d'abord : `Exécution → Modifier le type d'exécution → GPU`

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU détecté : {torch.cuda.get_device_name(0)}")
    print(f"   Mémoire disponible : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  Aucun GPU détecté — l'entraînement sera très lent sur CPU")
    print("   → Va dans : Exécution > Modifier le type d'exécution > GPU")


## 📁 Étape 1 — Montage Google Drive

Assure-toi que ton Drive contient cette structure **avant** d'exécuter cette cellule :
```
Mon Drive/
└── breast-cancer-classifier/
    ├── config.py
    ├── prepare_dataset.py
    ├── requirements.txt
    ├── src/
    │   ├── __init__.py
    │   ├── dataset.py
    │   ├── model.py
    │   ├── train.py
    │   ├── evaluate.py
    │   ├── gradcam.py
    │   └── predict.py
    ├── notebooks/
    │   ├── 01_exploration.ipynb
    │   ├── 02_preprocessing.ipynb
    │   ├── 03_training.ipynb
    │   ├── 04_evaluation.ipynb
    │   └── 05_gradcam.ipynb
    ├── data/
    │   ├── train/  (Benign/ + Malignant/)
    │   ├── val/    (Benign/ + Malignant/)
    │   └── test/   (Benign/ + Malignant/)
    └── models/     (dossier vide)
```


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive monté")


## 📂 Étape 2 — Configuration du chemin projet

In [ ]:
import os
import sys

# ✏️ Modifie ce chemin si ton dossier est ailleurs dans Drive
PROJECT_PATH = '/content/drive/MyDrive/breast-cancer-classifier'

os.chdir(PROJECT_PATH)
sys.path.insert(0, PROJECT_PATH)

print(f"📁 Dossier courant : {os.getcwd()}")
print()
print("Fichiers détectés :")
for f in sorted(os.listdir('.')):
    print(f"  {'📁' if os.path.isdir(f) else '📄'} {f}")


## 📦 Étape 3 — Installation des dépendances
> Durée : ~2 min (une seule fois par session Colab)

In [ ]:
%%capture install_output
!pip install torch torchvision --quiet
!pip install grad-cam opencv-python-headless --quiet
!pip install scikit-learn matplotlib pandas --quiet
!pip install fastapi uvicorn python-multipart --quiet
!pip install streamlit requests --quiet

print("✅ Toutes les dépendances installées")
import torch, torchvision, sklearn, cv2
print(f"   PyTorch     : {torch.__version__}")
print(f"   Torchvision : {torchvision.__version__}")
print(f"   Scikit-learn: {sklearn.__version__}")


In [ ]:
# Vérification finale de l'environnement
import torch
import numpy as np
import matplotlib
from pathlib import Path
from config import TRAIN_DIR, VAL_DIR, TEST_DIR, CLASS_NAMES, MODEL_PATH

print("✅ config.py importé avec succès")
print(f"   Classes     : {CLASS_NAMES}")
print(f"   Train dir   : {TRAIN_DIR}")
print(f"   Val dir     : {VAL_DIR}")
print(f"   Test dir    : {TEST_DIR}")
print()

# Vérification que les données sont bien là
for split, d in [("train", TRAIN_DIR), ("val", VAL_DIR), ("test", TEST_DIR)]:
    if d.exists():
        classes = [c.name for c in d.iterdir() if c.is_dir()]
        total = sum(len(list(c.iterdir())) for c in d.iterdir() if c.is_dir())
        print(f"  ✅ {split:5} : {total} images — classes : {classes}")
    else:
        print(f"  ❌ {split:5} : dossier introuvable → {d}")


## 🔍 Étape 4 — Exploration du dataset (Notebook 01)
> Distribution des classes, visualisation des images, détection d'images corrompues.


In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, UnidentifiedImageError
from config import TRAIN_DIR, CLASS_NAMES

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Distribution des classes ──────────────────────────────────
counts = {}
for cls_dir in sorted(TRAIN_DIR.iterdir()):
    if cls_dir.is_dir():
        n = len([f for f in cls_dir.iterdir()
                 if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}])
        counts[cls_dir.name] = n

total = sum(counts.values())
print(f"Total images (train) : {total}")
for cls, n in counts.items():
    print(f"  {cls:>12} : {n:>5} images ({100*n/total:.1f}%)")

if len(counts) == 2:
    vals = list(counts.values())
    ratio = max(vals) / min(vals)
    print(f"\nRatio déséquilibre : {ratio:.2f}:1")
    if ratio > 1.5:
        print("  ⚠️  Déséquilibre → class_weight activé dans la Loss")
    else:
        print("  ✅ Classes équilibrées")


In [ ]:
# ── Visualisation distribution ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
colors = ['#1D9E75', '#E24B4A']

bars = axes[0].bar(counts.keys(), counts.values(),
                   color=colors[:len(counts)], edgecolor='white', width=0.5)
axes[0].set_title("Nombre d'images par classe (train)", fontsize=12)
axes[0].set_ylabel("Nombre d'images")
for bar, (cls, n) in zip(bars, counts.items()):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 str(n), ha='center', va='bottom', fontweight='bold')

axes[1].pie(counts.values(), labels=counts.keys(),
            colors=colors[:len(counts)], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor':'white','linewidth':2})
axes[1].set_title('Proportion des classes', fontsize=12)

plt.suptitle('Distribution du dataset (split train)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('models/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Figure sauvegardée dans models/class_distribution.png")


In [ ]:
# ── Échantillon d'images par classe ──────────────────────────
fig, axes = plt.subplots(len(counts), 5, figsize=(16, 4*len(counts)))
if len(counts) == 1:
    axes = [axes]

for row, (cls_name, _) in enumerate(counts.items()):
    cls_dir = TRAIN_DIR / cls_name
    images  = [f for f in cls_dir.iterdir()
               if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}]
    sample  = random.sample(images, min(5, len(images)))
    for col, img_path in enumerate(sample):
        img = Image.open(img_path).convert('RGB')
        axes[row][col].imshow(img)
        axes[row][col].set_title(f'{cls_name}\n{img.size[0]}x{img.size[1]}', fontsize=8)
        axes[row][col].axis('off')

plt.suptitle("Échantillon d'images par classe (train)", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── Détection images corrompues ────────────────────────────────
corrupted = []
for cls_dir in TRAIN_DIR.iterdir():
    if not cls_dir.is_dir(): continue
    for f in cls_dir.iterdir():
        if f.suffix.lower() not in {'.jpg','.jpeg','.png','.bmp'}: continue
        try:
            with Image.open(f) as img:
                img.verify()
        except Exception as e:
            corrupted.append((f, str(e)))

print(f"Images corrompues : {len(corrupted)}")
for path, err in corrupted[:10]:
    print(f"  {path.name}: {err}")
if not corrupted:
    print("✅ Aucune image corrompue — dataset propre")


## 🔧 Étape 5 — Prétraitement & DataLoaders (Notebook 02)
> Transforms, augmentation, vérification batch, poids de classes.


In [ ]:
from src.dataset import (get_train_transform, get_val_transform,
                          get_dataloaders, compute_class_weights)
from torchvision import datasets
from config import TRAIN_DIR, VAL_DIR, TEST_DIR, BATCH_SIZE, IMG_SIZE, MEAN, STD, SEED
import torch, numpy as np

torch.manual_seed(SEED)
np.random.seed(SEED)

# Affichage des transforms
train_tf = get_train_transform()
val_tf   = get_val_transform()

print("=== Train transform (avec augmentation) ===")
for t in train_tf.transforms:
    print(f"  {t}")
print("\n=== Val/Test transform (sans augmentation) ===")
for t in val_tf.transforms:
    print(f"  {t}")


In [ ]:
# Chargement des DataLoaders
train_loader, val_loader, test_loader, class_names = get_dataloaders(
    batch_size=BATCH_SIZE,
    num_workers=2,
    use_weighted_sampler=True,
)

train_ds = datasets.ImageFolder(TRAIN_DIR, transform=get_train_transform())
val_ds   = datasets.ImageFolder(VAL_DIR,   transform=get_val_transform())
test_ds  = datasets.ImageFolder(TEST_DIR,  transform=get_val_transform())

print(f"Classes : {class_names}")
print(f"Index   : {train_ds.class_to_idx}")
print(f"\nSplits :")
print(f"  Train : {len(train_ds):>5} images  ({len(train_loader)} batches de {BATCH_SIZE})")
print(f"  Val   : {len(val_ds):>5} images  ({len(val_loader)} batches)")
print(f"  Test  : {len(test_ds):>5} images  ({len(test_loader)} batches)")


In [ ]:
# Vérification batch + visualisation
import matplotlib.pyplot as plt

images, labels = next(iter(train_loader))
print(f"Shape batch : {images.shape}  → [batch, canaux, H, W]")
print(f"Min/Max après normalisation : {images.min():.2f} / {images.max():.2f}")

mean = np.array(MEAN)
std  = np.array(STD)

def denorm(t):
    img = t.permute(1, 2, 0).numpy()
    return (img * std + mean).clip(0, 1)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for i, ax in enumerate(axes.flat):
    if i >= len(images): break
    ax.imshow(denorm(images[i]))
    label = class_names[labels[i].item()]
    ax.set_title(label, fontsize=10,
                 color='#A32D2D' if label == 'Malignant' else '#085041')
    ax.axis('off')
plt.suptitle('Batch train après augmentation (dénormalisé)', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
# Poids de classes pour la Loss
class_weights = compute_class_weights(train_ds)
print("Poids de classes :")
for cls, w in zip(class_names, class_weights.tolist()):
    print(f"  {cls:>12} : {w:.4f}")
print("\n✅ Ces poids seront passés à CrossEntropyLoss")


## 🚀 Étape 6 — Entraînement EfficientNet-B3 (Notebook 03)
> Transfer learning + fine-tuning + early stopping  
> ⏱️ Durée estimée : **15–25 min** sur GPU T4


In [ ]:
from src.model import build_model, count_parameters

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")

model = build_model(freeze_backbone=True)
params = count_parameters(model)
print(f"\nParamètres totaux      : {params['total']:,}")
print(f"Paramètres entraînables : {params['trainable']:,}")
print(f"Backbone gelé           : {100*(params['total']-params['trainable'])/params['total']:.1f}%")
print("\nArchitecture tête custom :")
print(model.classifier)


In [ ]:
from src.train import train, set_seed
from config import NUM_EPOCHS, LR, MODEL_PATH

set_seed(42)

print("Lancement de l'entraînement...")
print(f"  Epochs max    : {NUM_EPOCHS}")
print(f"  LR initial    : {LR}")
print(f"  Fine-tuning   : dégel backbone après époque 5")
print(f"  Early stopping: patience = 7 époques")
print(f"  Sauvegarde    : {MODEL_PATH}")
print()

history = train(
    num_epochs=NUM_EPOCHS,
    lr=LR,
    unfreeze_after=5,
)

print(f"\n✅ Entraînement terminé — modèle sauvegardé : {MODEL_PATH}")


In [ ]:
# Courbes d'apprentissage
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train', color='#7F77DD', lw=2)
axes[0].plot(history['val_loss'],   label='Val',   color='#D85A30', lw=2)
axes[0].set_title('Loss', fontsize=13)
axes[0].set_xlabel('Époque')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', color='#7F77DD', lw=2)
axes[1].plot(history['val_acc'],   label='Val',   color='#D85A30', lw=2)
axes[1].set_title('Accuracy', fontsize=13)
axes[1].set_xlabel('Époque')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("Courbes d'apprentissage", fontsize=14)
plt.tight_layout()
plt.savefig('models/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

best_epoch = history['val_acc'].index(max(history['val_acc'])) + 1
print(f"Meilleure val accuracy : {max(history['val_acc']):.4f} (époque {best_epoch})")
print("✅ Courbes sauvegardées dans models/training_curves.png")


## 📊 Étape 7 — Évaluation complète (Notebook 04)
> Matrice de confusion · AUC-ROC · F1-score · Analyse des erreurs


In [ ]:
from src.model import load_model
from src.evaluate import (get_predictions, plot_confusion_matrix,
                           plot_roc_curve, print_classification_report,
                           find_misclassified)
from config import MODEL_PATH, CLASS_NAMES

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = load_model(MODEL_PATH, device)
_, _, test_loader, class_names = get_dataloaders(num_workers=2)

print("✅ Modèle chargé — calcul des prédictions sur le test set...")
labels, preds, probs = get_predictions(model, test_loader, device)
print(f"Nombre d'images évaluées : {len(labels)}")


In [ ]:
# Rapport de classification
print_classification_report(labels, preds, class_names)


In [ ]:
# Matrice de confusion
plot_confusion_matrix(labels, preds, class_names,
                      save_path='models/confusion_matrix.png')
print("✅ Matrice sauvegardée dans models/confusion_matrix.png")


In [ ]:
# Courbe ROC — métrique principale en médical
roc_auc = plot_roc_curve(labels, probs,
                          save_path='models/roc_curve.png')
print(f"\nAUC-ROC = {roc_auc:.4f}")
if roc_auc >= 0.90:
    print("✅ Excellent (AUC ≥ 0.90)")
elif roc_auc >= 0.85:
    print("✅ Bon (AUC ≥ 0.85)")
else:
    print("⚠️  À améliorer (AUC < 0.85)")
print("✅ Courbe ROC sauvegardée dans models/roc_curve.png")


In [ ]:
# Analyse des images mal classées
print("Images mal classées (faux positifs / faux négatifs) :")
find_misclassified(model, test_loader, device, class_names, n_show=8)


In [ ]:
# Synthèse finale des métriques
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

accuracy    = accuracy_score(labels, preds)
f1_macro    = f1_score(labels, preds, average='macro')
f1_weighted = f1_score(labels, preds, average='weighted')

fn = ((labels == 1) & (preds == 0)).sum()
fp = ((labels == 0) & (preds == 1)).sum()

print("=" * 50)
print("MÉTRIQUES FINALES — SET DE TEST")
print("=" * 50)
print(f"Accuracy    : {accuracy:.4f} ({accuracy:.1%})")
print(f"AUC-ROC     : {roc_auc:.4f}")
print(f"F1 Macro    : {f1_macro:.4f}")
print(f"F1 Weighted : {f1_weighted:.4f}")
print(f"\nFaux négatifs (Malignant→Benign) : {fn}  ⚠️  Critique")
print(f"Faux positifs  (Benign→Malignant) : {fp}")
print("=" * 50)


## 🔥 Étape 8 — Explicabilité Grad-CAM (Notebook 05)
> Visualisation des zones décisives du modèle — point fort pour la note ⭐


In [ ]:
try:
    from pytorch_grad_cam import GradCAM
    print("✅ pytorch-grad-cam déjà installé")
except ImportError:
    !pip install grad-cam -q
    print("✅ pytorch-grad-cam installé")


In [ ]:
from src.gradcam import visualize_gradcam, batch_gradcam

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = load_model(MODEL_PATH, device)
_, _, test_loader, class_names = get_dataloaders(num_workers=2)

# Sélection d'une image Benign et une Malignant
benign_t, malignant_t = None, None
for imgs, lbls in test_loader:
    for i in range(len(lbls)):
        if lbls[i].item() == 0 and benign_t is None:
            benign_t = imgs[i:i+1]
        if lbls[i].item() == 1 and malignant_t is None:
            malignant_t = imgs[i:i+1]
    if benign_t is not None and malignant_t is not None:
        break

print("Grad-CAM — image Benign :")
visualize_gradcam(model, benign_t.to(device), device, class_names, title="Exemple Benign")

print("Grad-CAM — image Malignant :")
visualize_gradcam(model, malignant_t.to(device), device, class_names, title="Exemple Malignant")


In [ ]:
# Grille complète 6 images
batch_gradcam(
    model, test_loader, device,
    class_names=class_names,
    n_samples=6,
    save_path='models/gradcam_grid.png',
)
print("✅ Grille Grad-CAM sauvegardée dans models/gradcam_grid.png")


## ✅ Étape 9 — Résumé final & vérification des artefacts

In [ ]:
from pathlib import Path

print("=" * 55)
print("PROJET COMPLET — ARTEFACTS GÉNÉRÉS")
print("=" * 55)

artefacts = [
    ("models/best_model.pth",          "Modèle entraîné"),
    ("models/class_names.json",        "Noms des classes"),
    ("models/transform_config.json",   "Config preprocessing"),
    ("models/training_curves.png",     "Courbes d'apprentissage"),
    ("models/class_distribution.png",  "Distribution classes"),
    ("models/confusion_matrix.png",    "Matrice de confusion"),
    ("models/roc_curve.png",           "Courbe ROC"),
    ("models/gradcam_grid.png",        "Grille Grad-CAM"),
]

for path, desc in artefacts:
    exists = Path(path).exists()
    size   = f"({Path(path).stat().st_size / 1e6:.1f} MB)" if exists and Path(path).suffix == '.pth' else ""
    status = "✅" if exists else "❌"
    print(f"  {status} {desc:<30} {path} {size}")

print()
print("Métriques finales :")
print(f"  Accuracy  : {accuracy:.4f}")
print(f"  AUC-ROC   : {roc_auc:.4f}")
print(f"  F1 Macro  : {f1_macro:.4f}")
print()
print("Prochaine étape : lancer l'API FastAPI + Streamlit")
print("  → uvicorn api.main:app --port 8000")
print("  → streamlit run app/app.py")


## 🌐 Bonus — Tester l'API FastAPI dans Colab
> Optionnel — permet de vérifier que l'API fonctionne avant le déploiement cloud.


In [ ]:
# Lancement de l'API en arrière-plan
import subprocess, time, requests

proc = subprocess.Popen(
    ["uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(5)  # attendre le démarrage

# Vérification du health check
try:
    r = requests.get("http://localhost:8000/health", timeout=5)
    health = r.json()
    print("✅ API démarrée avec succès")
    print(f"   Status       : {health['status']}")
    print(f"   Modèle chargé: {health['model_loaded']}")
    print(f"   Device       : {health['device']}")
    print(f"   Classes      : {health['class_names']}")
except Exception as e:
    print(f"❌ Erreur API : {e}")


In [ ]:
# Test d'une prédiction via l'API
import io
from PIL import Image

# Prend la première image du test set
test_images = list(next(iter(
    Path('data/test').rglob('*.jpg')
) if list(Path('data/test').rglob('*.jpg')) else
    Path('data/test').rglob('*.png')
).__class__.__mro__[0].rglob(Path('data/test'), '*.jpg'))  # fallback

# Méthode directe
from pathlib import Path
test_img_path = next(
    p for split in ['Benign','Malignant']
    for p in (Path('data/test') / split).iterdir()
    if p.suffix.lower() in {'.jpg','.jpeg','.png'}
)

print(f"Image de test : {test_img_path}")

with open(test_img_path, 'rb') as f:
    response = requests.post(
        "http://localhost:8000/predict",
        files={"file": (test_img_path.name, f, "image/jpeg")},
        timeout=30,
    )

result = response.json()
print(f"\n✅ Réponse API :")
print(f"   Classe prédite  : {result['predicted_class']}")
print(f"   Confiance       : {result['confidence']:.1%}")
print(f"   Probabilités    : {result['probabilities']}")
print(f"   Temps inférence : {result.get('inference_time_ms','?')} ms")
print(f"   Grad-CAM        : {'✅ présent' if result.get('gradcam_base64') else '❌ absent'}")

# Arrêt de l'API
proc.terminate()
print("\nAPI arrêtée.")
